In [1]:
from openai import OpenAI
import os 
from dotenv import load_dotenv
from minsearch import Index
from pathlib import Path
import onnxruntime as ort
from tokenizers import Tokenizer
import numpy as np
from gitsource import chunk_documents
from tqdm.auto import tqdm
import numpy as np

load_dotenv()

client = OpenAI()

In [ ]:
# !uv run python download.py

tokenizer.json: 100%|███████████████████████| 712k/712k [00:00<00:00, 43.6MB/s]
  saved models/Xenova/all-MiniLM-L6-v2/tokenizer.json
onnx/model.onnx: 100%|████████████████████| 90.4M/90.4M [00:04<00:00, 18.5MB/s]
  saved models/Xenova/all-MiniLM-L6-v2/model.onnx


In [ ]:
# Q1. Embedding a query

from embedder import Embedder

embed = Embedder()

q = "How does approximate nearest neighbor search work?"
v = embed.encode(q)

v[0]

np.float64(-0.02058203437252893)

In [ ]:
# Q2. Cosine similarity
# 
# Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, 
# embed its content, and compute the cosine similarity with the query vector from Q1. 
# What do you get?

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [ ]:
content = None
for dict_ in documents:
    if dict_["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md":
        content = dict_["content"]
        break
content    

'# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done so far is ex

In [11]:
q2 = content
v2 = embed.encode(q2)

v2.dot(v)

np.float64(0.36107027225589694)

In [33]:
q_temp = "Let's use VectorSearch from minsearch"
v_temp = embed.encode(q_temp)

scores = X.dot(v)

idx = np.argmax(scores)
chunks[idx]


{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

In [ ]:
# Q3. Chunking and search by hand
# 
# Q3 - Which file does the highest-scoring chunk belong to (its filename)?

chunks = chunk_documents(documents, size=2000, step=1000)
texts = [c["content"] for c in chunks]

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)
scores = X.dot(v)

idx = np.argmax(scores)
chunks[idx]

  0%|          | 0/6 [00:00<?, ?it/s]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

In [ ]:
# Q4. Vector search with minsearch
# 
# Which file is the filename of the first result?

from minsearch import VectorSearch

texts = [doc["content"] for doc in documents]

batch_size = 50
X2 = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X2.extend(batch_vectors)

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X2, documents)

  0%|          | 0/2 [00:00<?, ?it/s]

In [41]:
q3 = "What metric do we use to evaluate a search engine?"
v3 = embed.encode(q3)

results = vindex.search(v3, num_results=5)
results

[{'content': '# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup, each query

In [ ]:
# Q5. Text search vs vector search
# 
# Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

q5 = "How do I store vectors in PostgreSQL?"

# vector
v5 = embed.encode(q5)
results_v = vindex.search(v5, num_results=5)


index = Index(
    text_fields=['content']
)
index.fit(documents)

results_i = index.search(
    q5,
    num_results=5
)

# ['02-vector-search/lessons/08-pgvector.md\n' is the only one 
# in the vector search results that is not in the text search results]
print("Vector search results:")
display([f"{elem["filename"]}\n" for elem in results_v])
print("")
print("Text search results:")
display([f"{elem["filename"]}\n" for elem in results_i])

Vector search results:


['02-vector-search/lessons/08-pgvector.md\n',
 '05-monitoring/lessons/05-database.md\n',
 '02-vector-search/lessons/02-embeddings.md\n',
 '02-vector-search/lessons/04-vector-search.md\n',
 '02-vector-search/lessons/07-sqlitesearch-vector.md\n']


Text search results:


['02-vector-search/lessons/02-embeddings.md\n',
 '02-vector-search/lessons/01-intro.md\n',
 '02-vector-search/lessons/03-embeddings-dataset.md\n',
 '03-orchestration/lessons/05-rag.md\n',
 '02-vector-search/lessons/10-next-steps.md\n']

In [ ]:
# Q6. Hybrid search

# Which file is ranked first after RRF?
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

q6 = "How do I give the model access to tools?"

# vector
v6 = embed.encode(q6)
results_v = vindex.search(v6, num_results=30)

# text
results_i = index.search(
    q6,
    num_results=30
)

results = rrf([results_v, results_i])

print("Vector results:")
display([f"{elem['filename']}\n" for elem in results_v[:10]])
print("-------")
print("Text results:")
display([f"{elem['filename']}\n" for elem in results_i[:10]])
print("-------")
print("RRF results:")
display([f"{elem['filename']}\n" for elem in results[:10]])



Vector results:


['01-agentic-rag/lessons/16-other-frameworks.md\n',
 '05-monitoring/lessons/02-assistant-setup.md\n',
 '01-agentic-rag/lessons/07-llm.md\n',
 '07-project-example/lessons/04-interface.md\n',
 '04-evaluation/lessons/11-evaluation-intro.md\n',
 '05-monitoring/lessons/03-chat-app.md\n',
 '03-orchestration/lessons/06-agents.md\n',
 '03-orchestration/lessons/08-best-practices.md\n',
 '01-agentic-rag/lessons/06-building-prompt.md\n',
 '03-orchestration/lessons/09-next-steps.md\n']

-------
Text results:


['01-agentic-rag/lessons/13-function-calling.md\n',
 '01-agentic-rag/lessons/14-agentic-loop.md\n',
 '01-agentic-rag/lessons/11-agents-intro.md\n',
 '01-agentic-rag/lessons/15-frameworks.md\n',
 '05-monitoring/lessons/14-next-steps.md\n',
 '01-agentic-rag/lessons/03-rag.md\n',
 '01-agentic-rag/lessons/07-llm.md\n',
 '01-agentic-rag/lessons/01-intro.md\n',
 '01-agentic-rag/lessons/16-other-frameworks.md\n',
 '04-evaluation/lessons/13-llm-as-judge.md\n']

-------
RRF results:


['01-agentic-rag/lessons/16-other-frameworks.md\n',
 '01-agentic-rag/lessons/07-llm.md\n',
 '01-agentic-rag/lessons/11-agents-intro.md\n',
 '04-evaluation/lessons/11-evaluation-intro.md\n',
 '01-agentic-rag/lessons/15-frameworks.md\n']